

# Retail Customer Intelligence — Scenariusz warsztatowy
## Pełen case study: od surowych danych przez zabezpieczone AI do Agent App

---

### Historia biznesowa

Jesteś **analitykiem danych** w firmie **TechRetail Corp** — dystrybutorze elektroniki
użytkowej B2B. Firma sprzedaje produkty marek takich jak Rony, Opple, Ramsung i Zamaha
do tysięcy firm w całych Stanach Zjednoczonych.

Zarząd przychodzi z trzema pytaniami:

> *„Kto są nasi najlepsi klienci? Kto może odejść? I jak możemy dać zespołowi
> sprzedaży narzędzie, które odpowie na te pytania w czasie rzeczywistym —
> ale NIE ujawni danych osobowych klientów?”*

Twoje zadanie: zbudować **kompletny pipeline analityczny** od surowych danych
po zabezpieczonego AI asystenta, przechodząc przez cztery warsztaty.

---

### Mapa 4 warsztatów

```
WS1: BUDOWA                 WS2: ZABEZPIECZENIE                 WS3: RAG                              WS4: AGENT APP
════════════════════════    ════════════════════════════════    ══════════════════════════════════    ═══════════════════════════════════
Marketplace (+ licencja)    Guardrails LLM: przykłady,          PDF docs (fpdf2)                      UC Functions (SQL+Py)
→ SQL                       system prompt, safety filter,       ↓                                     + test payloadem + AI Playground
↓                           własny guard (taksonomia S1–S6),    ai_parse_document() → metadane, bbox  ↓
AI Functions                AI Gateway (+ secret scope)         ↓                                     LangChain Agent + guardrails
↓                           ↓                                   Chunking (2000/200, == page ==)       ↓
Tool Calling (UC Func)                                                                                MCP Google Drive (Docs/Sheets)
↓                           ↓                                   ↓                                     ↓
PySpark → Gold Table -→     Guardrails UC (Row/Col)             ↓                                     MLflow Tracing (Workspace / UC)
↓                           ↓                                   Vector Search na chunkach:            ↓
ML + Registry @champion     Ewaluacja (Gold + Genie)            ANN / hybrid / full-text / filtry     UC Model @champion
↓                           + benchmark A/B (ROUGE, sędzia 1–5) / reranking / Playground              ↓
Dashboard + Genie Space     ↓                                   ↓                                     Model Serving + inference table
                            Monitoring danych (Snapshot)        RAG chain (LangChain) → UC @champion  ↓ batch ai_query
                            + monitoring odpowiedzi LLM         ↓                                     Databricks App (Gradio)
                              (Time Series 5 min)               Knowledge Assistant                   ↓
                                                                ↓                                     inference table → monitoring (WS2 §7)
                                                                Porównanie 3 RAGów
```

### Czas trwania

| Warsztat | Czas | Sekcje |
| --- | --- | --- |
| WS1: Budowa | \~105 min | 6 aktów (Marketplace + licencja → AI Functions → **Tool Calling** → Gold Table → ML → Dashboard + Genie) |
| WS2: Zabezpieczenie | \~115 min | 3 akty (Guardrails 4 warstwy → Ewaluacja + benchmark → Monitoring danych i odpowiedzi) |
| WS3: RAG i Knowledge Assistant | \~90 min | 4 części (Dokumenty → Custom RAG z chunkingiem i łańcuchem w UC → KA → Ewaluacja) |
| WS4: Agent App | \~120 min | 6 aktów (UC Functions → Agent + Tracing → **MCP Google Drive** → Rejestracja → Serving + batch + App + inference table) |
| **Razem** | **\~7h 10min** | Pełen pipeline: dane → zabezpieczenia → RAG → Agent App |

> **Skracanie:** sekcje oznaczone w notebookach jako *opcjonalne* (WS2 §7 AI Gateway, WS3 reranking i czyszczenie
> Markdown, WS4 trace'y w UC) można pominąć bez utraty ciągłości fabuły — razem ok. 40 min.

# Persona i kontekst startowy

> **Przeczytaj uczestnikom przed startem warsztatu**

---

**Kim jesteś:** Senior Data Analyst w TechRetail Corp, raportujesz do VP of Sales.

**Twój zespół:**
- Ty — analityk danych, budujesz pipeline i modele
- Sales Team — chcą dashboard i AI asystenta do odpytywania danych klientów
- Compliance Officer — wymaga żeby dane PII (tax_id, adresy) były chronione
- CTO — chce monitoring jakości danych i modeli + alerty na dryf

**Dane:** Masz dostęp do **Databricks Marketplace**, skąd pobierasz gotowy dataset
symulowanych klientów B2B elektroniki. Dane są realistyczne — zawierają:
- 28,813 klientów z pełnymi danymi adresowymi i PII
- 4,074 zamówień z danymi JSON o produktach i promocjach
- 360 rekordów sprzedaży per kategoria/miesiąc

**Cel końcowy:** Na koniec dnia masz działający system, który:
1. Klasyfikuje klientów w 4 segmenty lojalności (ML)
2. Daje zespołowi sprzedaży AI asystenta (Genie Space)
3. Chroni dane PII przed nieuprawnionym dostępem (guardrails)
4. Automatycznie monitoruje jakość danych i modelu (Lakehouse Monitoring)
5. Ewaluuje czy AI asystent działa poprawnie (MLflow GenAI Eval)



# WARSZTAT 1: Od danych z Marketplace do AI

**Notebook:** *Retail Forecasting Workshop od danych do AI*

---

## Akt 1: Poznanie danych (Sekcje 1-2, \~25 min)

### Kontekst fabularny
> *VP of Sales dzwoni: „Mamy nowy dataset z Marketplace — 28 tysięcy klientów B2B.
> Zanim cokolwiek zbudujemy, muszę rozumieć co mamy. Ile klientów per segment?
> Jakie są trendy zamówień? Kto jest naszym top klientem?"*
>
> *Compliance Officer dodaje: „I zanim to wgracie — na jakich warunkach używamy tych danych?”*

### Co robimy
| Komórka | Co | Czego szukamy |
| --- | --- | --- |
| 3 | **Przegląd warunków licencji** datasetu (rekord do audytu) | Dostawca, warunki, zakres, PII — zapisane |
| 4 | `DESCRIBE CATALOG EXTENDED` — katalog Delta Sharing | Catalog Type, Provider, Share — UC zna pochodzenie danych |
| 6 | Przegląd 3 tabel Marketplace | 28K klientów, 4K zamówień, 360 sprzedaży |
| 7 | Rozkład segmentów per stan | Segmenty 0-3 nieściśle rozmieszczone geograficznie |
| 9 | Trend tygodniowy zamówień | Pik w październiku 2019, spadek w listopadzie |
| 10 | Ranking klientów per segment | Top 3 per segment — VIP mają 50-60+ zamówień |

### Punkt dyskusji
> *„Zobaczcie że segment 3 (VIP) ma średnią monetary ponad 1000$ — 100x więcej niż segment 0.
> To właśnie ci klienci, których chcemy chronić i monitorować w Warsztacie 2."*

> *„Katalog Marketplace jest read-only. Wszystko, co z niego zbudujemy, to NASZA kopia —
> i to ona podlega naszym zasadom governance. Zapamiętajcie to na WS2."*

### Oczekiwane wyniki
- ✅ Rekord przeglądu licencji uzupełniony
- ✅ 3 tabele załadowane z Marketplace
- ✅ Wizualizacja słupkowa segmentów
- ✅ Trend WoW z growth rate
- ✅ Top 3 klienci per segment



## Akt 2: AI bezpośrednio w SQL (Sekcja 3, \~10 min)

### Kontekst fabularny
> *VP: „Fajne wykresy, ale potrzebuję rekomendacji — co ZROBIĆ z każdym segmentem?
> I chcę automatyczną klasyfikację typów klientów, nie ręczną."*

### Co robimy
| Komórka | Co | „Wow effect” |
| --- | --- | --- |
| 12 | `ai_query()` — LLM generuje rekomendacje per segment | AI odpowiada po polsku, w kontekście retail |
| 13 | `ai_classify()` — automatyczna kategoryzacja klientów | 10 klientów → 4 typy (high_value, growing, at_risk, dormant) |

### Punkt dyskusji
> *„Zwróćcie uwagę: AI Functions działają bezpośrednio w SQL — nie potrzeba Pythona,
> API keys ani infrastruktury. Ale AI nie ma dostępu do danych — my przekazujemy
> mu kontekst w prompcie. To ważne dla bezpieczeństwa, które omówimy w WS2."*
>
> *„Ten sam `ai_query()` w WS2 porównamy z tańszym modelem (benchmark), a w WS4 zawołamy nim
> NASZ WŁASNY endpoint agenta."*

### Oczekiwane wyniki
- ✅ 4 rekomendacje biznesowe (po polsku)
- ✅ 10 klientów sklasyfikowanych jako high_value_loyal (wszyscy z seg. 3)

## Akt 2b: Tool Calling — LLM wywołuje nasze funkcje (Sekcja 3b, \~10 min)

### Kontekst fabularny
> *VP: "ai_query() jest fajne, ale to JA muszę napisać SQL i podać dane.
> A gdyby LLM SAM mógł sięgnąć po dane, których potrzebuje?"*

### Co to jest tool calling?
Różnica kluczowa:
- `ai_query()` — **my** wywołujemy LLM z danymi (Sekcja 3)
- **tool calling** — LLM **sam** wywołuje nasze funkcje, gdy uzna to za potrzebne

To fundamentalny mechanizm agentów AI — w WS4 zbudujemy pełnego agenta.

### Co robimy
| Komórka | Co | Wynik |
| --- | --- | --- |
| 20 | Markdown: wyjaśnienie tool calling vs ai_query, diagram flow | Kontekst |
| 21 | `CREATE FUNCTION workspace.default.get_revenue_summary(segment, state_filter)` + test | UC Function zwracająca przychód per segment/stan |
| 22 | Python: pytanie po polsku → LLM wybiera narzędzie → UC Function → LLM formatuje odpowiedź | Pełny flow tool calling z OpenAI API |

### Punkt dyskusji
> *"Zwróćcie uwagę: LLM sam zdecydował że potrzebuje get_revenue_summary
> i sam dobrał parametry (segment=3 → VIP, state_filter=NY).
> My tylko OPISALIŚMY funkcję — LLM zrobił resztę.
> W WS4 agent będzie miał wiele takich narzędzi i będzie je łączył."*

### Oczekiwane wyniki
- UC Function `get_revenue_summary` utworzona i testowalna z SQL
- LLM poprawnie wywołał narzędzie z parametrami segment=3, state_filter=NY
- Odpowiedź agenta: „Łączny przychód VIPów w NY wynosi $1,244,468”

## Akt 3: Od surowych JSONów do Gold Table (Sekcje 4-5, \~20 min)

### Kontekst fabularny
> *Data Engineer: „Żeby zbudować model ML, potrzebujemy czystej tabeli z cechami klientów.
> Dane zamówień są w JSON — trzeba je sparsować, policzyć RFM i połączyć z tabelą klientów."*

### Co robimy
| Komórka | Co | Kluczowe |
| --- | --- | --- |
| 15 | PySpark: parsowanie JSON, RFM, JOIN | 28,813 klientów × 19 kolumn |
| 17 | Zapis jako `gold_customer_360` w Delta Lake | ACID, Time Travel, wersjonowanie |
| 18-19 | DESCRIBE HISTORY + schemat | PII: tax_id, customer_name — zapamiętaj! |

### 🚨 Moment kluczowy — PII w danych
> *Compliance Officer patrzy na ekran: „Czekajcie — ta tabela ma tax_id i pełne nazwiska
> klientów? To są dane osobowe! Kto ma do nich dostęp?"*
>
> **To jest most do Warsztatu 2.** Zanotujcie: `gold_customer_360` zawiera PII.
> W WS2 zabezpieczymy ją przez column mask na `tax_id` i row filter na `state`.

### Oczekiwane wyniki
- ✅ Gold table: 28,813 wierszy, 19 kolumn
- ✅ Kolumny RFM: recency_days, frequency, monetary
- ✅ PII zidentyfikowane: tax_id, customer_name

## Akt 4: Model ML — kto jest VIP? (Sekcje 6-8, \~25 min)

### Kontekst fabularny
> *VP: „Chcę żeby system automatycznie przypisywał segment lojalności nowym klientom.
> Nie możemy czekać na ręczną analizę — potrzebuję modelu."*

### Co robimy
| Komórka | Co | Wynik |
| --- | --- | --- |
| 24 | Przygotowanie danych (stratified split) | 80/20, 4 klasy |
| 25 | Gradient Boosting + MLflow autolog | Accuracy \~0.90, F1 \~0.90 |
| 26 | Random Forest (porównanie) | Accuracy \~0.87, F1 \~0.87 |
| 28 | Rejestracja modelu w Unity Catalog **+ alias `@champion`** | `workspace.default.loyalty_segment_classifier@champion` |
| 30 | Batch inference na całej tabeli — model ładowany **przez alias** | Predykcja vs actual per klient |
| 32-33 | AutoML (opcjonalnie, ML Runtime) | Automatyczny dobór najlepszego algorytmu |

### Punkt dyskusji
> *„Model jest zarejestrowany w Unity Catalog z aliasem @champion — WS2 załaduje go
> po aliasie i sprawdzi, czy nadal dobrze działa na bieżących danych. Retraining = nowa wersja
> + przepięcie aliasu; nikt nie zmienia kodu konsumentów. Jeśli dane się zmienią (dryf!), accuracy
> spadnie. Dlatego potrzebujemy monitoringu."*

### Oczekiwane wyniki
- ✅ Dwa modele porównane w MLflow UI
- ✅ Model zarejestrowany w UC z aliasem `@champion`
- ✅ Batch inference z \~90% accuracy

## Akt 5: Dashboard i Genie — dajemy biznesowi narzędzia (Sekcje 9-10, \~15 min)

### Kontekst fabularny
> *VP: „Wszystko świetnie, ale ja nie umiem SQL. Potrzebuję dashboardu i asystenta,
> którego mogę po prostu zapytać po polsku: kto jest naszym najlepszym klientem w NY?"*

### Co robimy
| Komórka | Co | Wynik |
| --- | --- | --- |
| 35 | Prognoza przychodów (14 dni) | Random Forest na time features |
| 37 | Tworzenie Dashboard SDK | Widgety: segmenty, przychody, geografia, prognoza |
| 38 | Tworzenie Genie Space | AI asystent odpytujący `gold_customer_360` |

### 🚨 Moment kluczowy — most do WS2
> *Compliance Officer: „Czekajcie — Genie Space ma dostęp do tabeli z PII?
> Ktoś może zapytać o tax_id wszystkich klientów! Musimy to zabezpieczyć."*
>
> **To jest cliﬀhanger przed Warsztatem 2:**
> - Genie Space działa — ale czy jest bezpieczny?
> - Model jest zarejestrowany — ale co jeśli dane się zmienią?
> - Dashboard pokazuje dane — ale kto widzi co?

### Oczekiwane wyniki
- ✅ Dashboard z widgetami
- ✅ Genie Space „Retail Customer Intelligence Assistant”
- ✅ Prognoza przychodów na 14 dni

---

### ☕ PRZERWA (10 min)

**Podsumowanie WS1:** Zbudowaliśmy kompletny pipeline:
`Marketplace (+ licencja) → SQL → AI → PySpark → Gold Table → ML @champion → Dashboard + Genie`

**Otwarte pytania na WS2:**
1. Czy Genie Space nie ujawni PII klientów?
2. Co jeśli dane się zmienią i model straci dokładność?
3. Kto ma dostęp do danych i jak to kontrolujemy?

# WARSZTAT 2: Guardrails, Monitoring i Ewaluacja

**Notebook:** *Retail Workshop 2 Guardrails Monitoring Ewaluacja*

---

## Akt 1: Zabezpieczenie AI i danych (Części 1-2, \~50 min)

### Kontekst fabularny
> *Compliance Officer: „Przed wdrożeniem muszę mieć kilka poziomów ochrony:
> 1) AI nie może odpowiadać na pytania spoza domeny retail — i nie może dać się nabrać na „to tylko powieść”;
> 2) nasze WŁASNE reguły (PII, nieuczciwe praktyki) muszą być jawne i logowane;
> 3) zabezpieczenie ma działać nawet, gdy ktoś napisze nowy notebook i „zapomni” o guardrails;
> 4) nawet jeśli ktoś zapyta o dane — PII musi być maskowane."*

### Co robimy — Część 1: Guardrails LLM
| Komórka | Co | Zabezpieczenie |
| --- | --- | --- |
| 3–5 | Przykłady: odmowa, **jailbreak „fikcyjna narracja”**, system prompt zawężający domenę | Uczestnicy widzą wzorzec ZANIM uruchomią kod |
| 7 | System prompt z guardrails (SDK) | LLM ograniczony do domeny klientów B2B |
| 8 | **Test jailbreaku na żywo** — czy sam prompt wystarczył? | Heurystyka odmowy; dyskusja |
| 10 | Safety filter Databricks | Blokowanie szkodliwych zapytań (polityka platformy) |
| 12 | **Własny guard**: taksonomia S1–S3 → S1–S6, format Llama Guard | Werdykt `safe/unsafe` + kategorie do audytu |
| 13 | Integracja: guard **przed** (User) i **po** (Agent) wywołaniu LLM | Odpowiedź wraca tylko gdy oba testy są safe |
| 15 | Secret scope + PAT (getpass) | Endpoint uwierzytelnia się poza sesją notebooka |
| 16 | **AI Gateway**: endpoint `retail-guarded-llm` z Safety, PII block, invalid keywords, inference table | Polityka na endpoincie — niezależna od kodu klienta |
| 17 | 3 prompty przez endpoint z AI Gateway | Blokada = HTTP 400 z powodem, zanim model cokolwiek zobaczy |

### Co robimy — Część 2: Guardrails danych (UC)
| Komórka | Co | Zabezpieczenie |
| --- | --- | --- |
| 22 | SHOW GRANTS na tabeli | Kto ma SELECT/MODIFY |
| 26 | ROW FILTER na `state` | Analityk CA widzi tylko Kalifornię |
| 29 | COLUMN MASK na `tax_id` | Non-admins widzą `***MASKED***` |
| 33 | Czyszczenie (dla monitoringu) | Usuwamy filtry żeby monitoring widział pełne dane |

### Punkt dyskusji
> *„Zwróćcie uwagę na warstwowość zabezpieczeń:
> - Warstwa 1: System prompt (aplikacja) — najtańsza, najłatwiej obejść (§4b!)
> - Warstwa 2: Safety filter (platforma, polityka Databricks)
> - Warstwa 3: Własny guard (nasza taksonomia, audytowalny werdykt)
> - Warstwa 4: AI Gateway (polityka przypięta do endpointu — działa dla App z WS4 i każdego klienta)
> - Warstwa 5: Row filter + Column mask (dane)
> Każda warstwa działa niezależnie — jeśli jedna zawiedzie, pozostałe nadal chronią."*

### Oczekiwane wyniki
- ✅ LLM odmawia odpowiedzi na pytanie o alarm sklepowy
- ✅ Test „powieści” — dyskusja, czy prompt wystarczył
- ✅ Własny guard: `UNSAFE S2` dla alarmu, `UNSAFE S3` dla prośby o tax_id, `safe` dla pytania o VIP
- ✅ Endpoint `retail-guarded-llm` READY; prompt PII zablokowany przez AI Gateway (400)
- ✅ Row filter działa (symulacja widoku innego użytkownika)
- ✅ tax_id zamaskowany jako `***MASKED***`

> **Jeśli brak uprawnień do endpointów/sekretów:** §7 czytamy jako przykład; reszta WS2 działa bez niego
> (monitoring odpowiedzi w Akcie 3 użyje logu z benchmarku).

## Akt 2: Ewaluacja — czy to co zbudowaliśmy działa? (Część 3, \~30 min)

### Kontekst fabularny
> *CTO: „Zanim włączymy monitoring ciągły, muszę widzieć baseline — jak dobrze
> działa Gold Table? Jak dobrze odpowiada Genie Space? Mam twarde liczby?
> I jeszcze jedno: płacimy za Llama 70B — czy tańszy model nie wystarczy? Nie zgadujcie, zmierzcie."*

### Co robimy
| Komórka | Co | Oczekiwane |
| --- | --- | --- |
| 38 | **Gold Table quality** — 7 testów z expected values | 7/7 PASS |
| 40 | Genie predict_fn — auto-discovery Space ID | Połączenie z Genie z WS1 |
| 41 | **6 test caseów z expected answers** | VIP=9541, top state=NY, brak zam.=26862 |
| 42 | Scorery: Safety, Guidelines, no_pii_leak, correctness | 4 scorery zdefiniowane |
| 43 | `mlflow.genai.evaluate()` na Genie | safety=1.0, no_pii_leak=1.0, correctness\~0.83 |
| 45 | **Benchmark A/B** — zbiór 8 wierszy (4 segmenty × 2 stany) z `gold_customer_360` | Pytanie + `expected_facts` + zdanie referencyjne |
| 46 | Dwa systemy (Llama 70B vs GPT-OSS 20B) + scorery: **ROUGE-1**, **sędzia 1–5 z przykładami**, Guidelines | Identyczny prompt, różni się tylko endpoint |
| 47 | Uruchomienie + tabela porównawcza (**mean / variance / min**) + zapis logu inferencji | `retail_assistant_inference_log` |

### Punkt dyskusji
> *„Zwróćcie uwagę na scorer `no_pii_leak` — testuje dokładnie to samo co column mask
> z Części 2. Jeśli ktoś obejdzie maskę, scorer to złapie. To jest defense in depth."*

> *„Każdy test case ma `expected_response` — wiemy DOKŁADNIE co powinno wyjść,
> bo sami policyliśmy te wartości w Warsztacie 1. To nie jest \"mam nadzieję że AI
> odpowie dobrze\" — to jest weryfikowalne."*

> *„W benchmarku ROUGE jest niski dla OBU modeli — parafraza to nie referencja. Dlatego obok metryki
> leksykalnej mamy sędziego 1–5. I patrzcie na WARIANCJĘ: model ze średnią 4.0 i wariancją 1.5 czasem
> daje jedynki — w produkcji to gorsze niż stabilne 3.8."*

### Oczekiwane wyniki
- ✅ Gold table: 7/7 testów przeszło
- ✅ Genie: safety i no_pii_leak = 100%
- ✅ Correctness: \~83% (Genie poprawnie odpowiada na 5/6 pytań)
- ✅ Benchmark: tabela baseline vs challenger w MLflow (2 runy w `/Shared/retail_llm_benchmark_workshop`)
- ✅ Tabela `retail_assistant_inference_log` z 16 odpowiedziami (wejście do Aktu 3 §7)

## Akt 3: Monitoring — oczy na danych i na odpowiedziach 24/7 (Część 4, \~35 min)

### Kontekst fabularny
> *CTO: „Mamy baseline z ewaluacji. Teraz potrzebuję systemu, który automatycznie
> wykryje gdy dane się zmienią, model straci dokładność, albo ktoś zmieni uprawnienia.
> A skoro asystent rozmawia z ludźmi — chcę też wiedzieć, CO im odpowiada. Codziennie. Bez czytania logów."*

### Co robimy
| Komórka | Co | Wynik |
| --- | --- | --- |
| 50 | Inspekcja danych przed monitoringiem | 28,813 wierszy, 4 segmenty |
| 52 | Tworzenie monitora **Snapshot** (Lakehouse Monitoring SDK) | Slicing: loyalty_segment, state |
| 53 | Harmonogram refreshu (cron) | Codziennie 8:00 |
| 55 | Refresh monitora | \~10 min, status: SUCCESS |
| 57 | **Profil per segment** | min/max/avg monetary, % null tax_id |
| 59 | **Analiza dryfu** | Segment 3 (VIP): z-score = 5.36 → ⚠️ DRYF |
| 61 | Dashboard monitora + link do Catalog Explorer | Automatyczny dashboard Lakeview |
| 63 | Wyniki ewaluacji z Cz. 3 (MLflow) | Połączenie eval → monitoring |
| 64 | Jakość modelu ML z WS1 (ładowany przez **`@champion`**) | Accuracy/F1 na bieżących danych |
| 65–66 | **Audit guardrails** + SHOW GRANTS | Checklist produkcyjny |
| 68 | **§7a** Rozpakowanie payloadów AI Gateway (JSON request/response) + log offline z benchmarku | Jedna tabela odpowiedzi z dwóch źródeł |
| 69 | **§7b** Metryki per odpowiedź: toxicity (`ai_classify`), readability, refusal, długość, perplexity* — zapis **przyrostowy** | `retail_assistant_processed_inference` |
| 70 | **§7c** Monitor **Time Series 5 min** + refresh + profil/dryf metryk | Trend jakości odpowiedzi w oknach czasowych |

### Punkt dyskusji
> *„Segment 3 (VIP) ma z-score 5.36 — to poważny dryf! W produkcji to byłby
> automatyczny alert. Jeśli rozkład VIP się zmienia, model z WS1 też traci
> dokładność — dlatego monitoring danych i monitoring modelu idą w parze."*

> *„§7: odpowiedzi LLM to TEŻ dane. Snapshot pyta „jak wygląda cała tabela”, Time Series pyta
> „czy ostatnie 5 minut różni się od wcześniejszych”. Skok `is_refusal` = guardrails za ostre albo atak;
> skok `toxicity` = model zaczął mówić rzeczy, których nie powinien; spadek `answer_length` = model ucina."*

### Oczekiwane wyniki
- ✅ Monitor Snapshot aktywny z refreshem SUCCESS
- ✅ Dryf wykryty (segment 3)
- ✅ Model quality check przez `@champion`
- ✅ Audit guardrails z checklistem produkcyjnym
- ✅ Tabela `retail_assistant_processed_inference` z metrykami; monitor Time Series utworzony (refresh może trwać)



# WARSZTAT 3: RAG i Knowledge Assistant

**Notebook:** [Retail Workshop 3 RAG i Knowledge Assistant](#notebook-3781019916085186)

---

## Kontekst fabularny
> *VP of Sales: "Dashboard i Genie Space są super, ale mój zespół chce też
> chatbota, który odpowiada na pytania kontekstowe — nie tylko liczbowe.
> Czy można stworzyć asystenta, który czyta nasze raporty i odpowiada z cytatami?"*

## Część 1: Generowanie dokumentów (~10 min)

| Komórka | Co | Wynik |
| --- | --- | --- |
| 5 | Statystyki per segment, top stany, ogólne | Input do raportów |
| 6–7 | `fpdf2` + `matplotlib` — **10 artykułów PDF** (3 strony: tabela, wykres, wnioski) | Dokumenty z danych strukturalnych |
| 8 | Zapis PDF do UC Volume `retail_docs` | Dokumenty gotowe do parsowania |

### Punkt dyskusji
> *"W realnych projektach RAG dokumenty to PDF, nie tekst. Dlatego generujemy PDF-y z tabelami i wykresami —
> i zaraz zobaczymy, ile z tej struktury parser potrafi odzyskać."*

## Część 2: Custom RAG z Vector Search (~45 min)

| Komórka | Co | Wynik |
| --- | --- | --- |
| 10 | `ai_parse_document` (2.0, **obrazy stron, opisy figur**) → `retail_rag_docs` z pełnym JSON | 10 dokumentów sparsowanych |
| 12 | **Metadane parsowania**: strony, elementy, tabele/figury/nagłówki per dokument | Rozkład typów elementów |
| 13 | **Render strony z ramkami bbox** (SVG na obrazie strony) | „Wow effect”: parser widzi tabelę i wykres |
| 15 | Tekst per strona z separatorem `== page ==` | Wejście do chunkingu |
| 16 | *(opcjonalnie)* czyszczenie JSON → Markdown przez `ai_query` | Tabele zachowane jako Markdown |
| 17 | **Chunking** `RecursiveCharacterTextSplitter` 2000/200 → `retail_rag_chunks` (PK chunk_id, CDF) | ~30–60 chunków zamiast 10 dokumentów |
| 19 | Embedding „na piechotę” + podobieństwo kosinusowe | Parafraza bliżej niż zupa pomidorowa |
| 20 | Vector Search endpoint + Delta Sync index **na chunkach** (gte-large-en) | Index READY |
| 21 | Custom RAG: retrieve chunks + `ai_query()` (prompt jako parametr) | 6 pytań testowych z cytatami [doc #chunk] |
| 23 | **ANN vs HYBRID vs FULL_TEXT** + filtr `filters_json` po dokumencie | Różne chunki i score dla tego samego pytania |
| 25 | Reranking *(opcjonalnie)* | Kolejność przed/po |
| 26 | AI Playground: indeks jako narzędzie (UI) | Chunki i cytaty bez kodu |
| 28 | **Łańcuch LangChain** + MLflow Tracing (retriever → prompt → LLM) | Trace w `/Shared/retail_rag_workshop` |
| 29 | `mlflow.langchain.log_model` (models from code, resources) → **UC `retail_rag_chain@champion`** | Artefakt gotowy do wdrożenia jak agent w WS4 |

### Punkt dyskusji
> *"To jest pełny pipeline RAG — parsowanie, chunking, embedding, retrieval, generation. Widzicie każdy krok
> i możecie go zmienić: rozmiar chunka, tryb wyszukiwania, filtr. Teraz zobaczmy jak Knowledge Assistant
> robi to samo automatycznie — w jednym wywołaniu API."*

## Część 3: Knowledge Assistant (~15 min)

| Komórka | Co | Wynik |
| --- | --- | --- |
| 31 | Tworzenie Knowledge Assistant (API Agent Bricks) + źródło Volume + sync | Agent ACTIVE z instrukcjami + guardrails PII |
| 32 | Quality examples (9 przykładów z guidelines) | Poprawa jakości odpowiedzi |

### Punkt dyskusji
> *"Knowledge Assistant automatycznie robi chunking i embedding — nie musicie
> budować pipeline Vector Search. To najprostsza droga do RAG w Databricks.
> Ale nie wybierzecie rozmiaru chunka, trybu wyszukiwania ani filtrów — to cena za zero kodu."*

## Część 4: Porównanie i ewaluacja (~20 min)

| Komórka | Co | Wynik |
| --- | --- | --- |
| 34 | Odpytanie KA tymi samymi pytaniami co Genie (WS2) | 6 pytań testowych |
| 35–36 | `mlflow.genai.evaluate()` — te same scorery co WS2 | Safety, Guidelines, PII, Correctness |
| 37 | Porównanie: Genie Space (SQL) vs Custom RAG vs Knowledge Assistant | Tabela wyników |
| 39 | Interaktywny widget — oba RAG-i z jednego miejsca | Demo na życzenie uczestników |

### 🚨 Moment kluczowy — Genie vs KA
> *"Genie Space odpowiada precyzyjnie (9541 klientów), bo generuje SQL.
> Knowledge Assistant odpowiada kontekstowo ('ponad 9500 klientów...'),
> bo czyta dokumenty. Oba chronią PII — różnymi mechanizmami.
> Który interfejs lepszy? Zależy od pytania!"*

### Oczekiwane wyniki
- 10 artykułów PDF w UC Volume; render strony z ramkami elementów
- Tabela `retail_rag_chunks` i indeks `retail_rag_chunks_index` ONLINE
- Porównanie trybów ANN/HYBRID/FULL_TEXT; filtr po dokumencie
- Model `workspace.default.retail_rag_chain@champion` w Unity Catalog z trace'ami w MLflow
- Knowledge Assistant ACTIVE z 9 quality examples
- Ewaluacja: safety=100%, no_pii_leak=100%
- Tabela porównawcza Genie vs Custom RAG vs KA

# WARSZTAT 4: Od UC Functions do AI Agent App

**Notebook:** [Retail Workshop 4 Agent App](#notebook-3781019916085187)

---

## Kontekst fabularny
> *CTO: "Mamy zabezpieczone dane, monitoring działa. Teraz pokażcie jak zbudować
> prawdziwą AI aplikację — agenta, który używa NASZYCH danych klientów jako narzędzi,
> respektuje guardrails PII z WS2, loguje każdą interakcję i jest zarejestrowany w Unity Catalog."*

**Ciągłość:** agent działa na tabeli `workspace.default.gold_customer_360` z WS1. Żadnej zmiany datasetu —
ten sam klient, którego profil oglądaliśmy w WS1, staje się odpowiedzią agenta w Databricks App.

---

## Akt 1: Dane i konfiguracja (~10 min)

### Co robimy
| Komórka | Co | Kluczowe |
| --- | --- | --- |
| 2 | Instalacja `unitycatalog-ai`, `databricks-langchain`, `mlflow`; restart Pythona | Zależności przypięte do wersji |
| 4 | Konfiguracja: katalog, schema, Volume `singleapp`, eksperyment MLflow | `workspace.default` |
| 5 | Przegląd `gold_customer_360` | 19 kolumn, w tym PII (tax_id) — agent go NIE zobaczy |

### Oczekiwane wyniki
- Pakiety zainstalowane, Python zrestartowany
- Tabela Gold widoczna (28 813 klientów)
- Konfiguracja eksperymentu MLflow

## Akt 2: UC Functions jako narzędzia agenta (~25 min)

### Kontekst fabularny
> *"Agent AI potrzebuje narzędzi. W Databricks narzędziami agenta są funkcje
> Unity Catalog — SQL i Python. Agent nie czyta tabel bezpośrednio —
> wywołuje zarejestrowane funkcje, które zwracają dane. Jeśli funkcja nie zwraca tax_id — agent nie ma czego ujawnić."*

### Co robimy
| Komórka | Co | Typ |
| --- | --- | --- |
| 7 | `get_average_customer_value(segment)` — średnia monetary per segment (-1 = wszystkie) | SQL → DOUBLE |
| 8 | `get_customer_profile(customer_id)` — profil **bez PII** (tax_id, lat/lon) | SQL → STRING |
| 9 | `format_customer_for_agent(...)` — formatowanie tekstowe | Python UDF |
| 10 | **Test surowym payloadem** (`client.execute_function`) — co dokładnie zobaczy agent | 4 wywołania bez LLM |
| 11 | **AI Playground**: funkcje UC jako Tools (UI) | Wywołanie narzędzia z parametrami widoczne w panelu |

### Zasada architektury
> *"Python UDF w Unity Catalog NIE może czytać Delta table przez spark.sql().
> Dlatego odczyt danych realizuje funkcja SQL (get_customer_profile),
> a Python UDF tylko formatuje przekazane pola.
> Podział odpowiedzialności: SQL = dostęp do danych, Python = logika biznesowa."*

> *"Test payloadem to jednostkowy test narzędzia. Jeśli LLM w Playground nie sięga po funkcję —
> poprawiacie COMMENT funkcji, nie prompt agenta."*

### Oczekiwane wyniki
- 3 funkcje zarejestrowane w Unity Catalog
- Każda przetestowana pojedynczo z oczekiwanym wynikiem (profil bez tax_id)
- Playground pokazuje wywołanie narzędzia i surowy wynik

## Akt 3: Budowa agenta i MLflow Tracing (~25 min)

### Kontekst fabularny
> *"Mamy narzędzia — teraz składamy agenta. LLM decyduje KTÓRE narzędzie wywołać
> i z jakimi parametrami. MLflow śledzi każdą interakcję: input, output, czas, tokeny.
> Compliance Officer: a te ślady chcę mieć tam, gdzie resztę danych — w Unity Catalog."*

### Co robimy
| Komórka | Co | Wynik |
| --- | --- | --- |
| 13 | UCFunctionToolkit + ChatDatabricks + AgentExecutor; system prompt z guardrails WS2 | Agent gotowy |
| 14 | Test: średnia VIP + profil klienta; **PII probe**; **out-of-domain** | Agent wywołuje narzędzia; odmawia PII i hackowania |
| 15 | `mlflow.langchain.autolog()` + trace tags + walidacja długości promptu | Tracing aktywny w MLflow UI |
| 17 | *(opcjonalnie)* **Trace'y w Unity Catalog** — eksperyment z `trace_location=UnityCatalog(...)` | Tabele `retail_agent_*` w `workspace.default` (lub czytelny powód braku) |

### Punkt dyskusji
> *"Agent sam planuje sekwencję wywołań:
> 1) get_average_customer_value(3) → średnia wartość VIP
> 2) get_customer_profile(id) → profil klienta bez PII
> 3) format_customer_for_agent → sformatowana odpowiedź
> 4) Składa końcowy tekst dla użytkownika
>
> To jest podejście agentic — LLM planuje i wykonuje, nie my.
> W MLflow UI zobaczycie pełen trace: każde wywołanie narzędzia,
> czas odpowiedzi, zużycie tokenów. A w UC — te same trace'y jako tabele: SQL, GRANT, retencja."*

### Oczekiwane wyniki
- Agent odpowiada na złożone pytania łącząc wiele narzędzi
- PII probe i out-of-domain zakończone odmową (guardrails z WS2 w promptcie + w narzędziach)
- Trace widoczny w MLflow UI (czas, tokeny, wywołania); opcjonalnie tabele trace'ów w UC
- Walidacja długości promptu działa

## Akt 3b: MCP Google Drive — agent + dokumenty (\~15 min)

### Kontekst fabularny
> *CTO: "Agent odpowiada na pytania w notebooku. Ale nasz zespół sprzedaży używa Google Docs.
> Czy agent może automatycznie stworzyć dokument z profilem klienta i wysłać go na Drive?"*

### Co to jest MCP?
**Model Context Protocol** — otwarty standard (łączenia agentów AI z zewnętrznymi narzędziami.
Databricks udostępnia gotowe MCP Services w `system.ai` — m.in. `google_drive` (Docs, Sheets, Slides).

### Co robimy
| Komórka | Co | Wynik |
| --- | --- | --- |
| 18 | Markdown — architektura MCP, wymagania | Kontekst |
| 19 | Diagnostyka: lista narzędzi MCP Google Drive — **uruchom PRZED agentem** | 13 narzędzi: search, create, edit, read, permissions... |
| 20 | `DatabricksMCPServer` + `DatabricksMultiServerMCPClient` → **16 narzędzi** (3 UC + 13 Google) | Agent LangGraph: pobiera profil klienta → tworzy Google Doc |

### Wymagania
- Admin: OAuth connection do Google w **AI Gateway → MCPs**
- Użytkownik: zalogowanie się do połączenia Google (jednorazowe)
- `GRANT EXECUTE` na `system.ai.google_drive`

### Punkt dyskusji
> *"MCP to standard — ten sam wzorzec działa z dowolnym serwisem: Slack, Jira, Confluence.
> `DatabricksMultiServerMCPClient` przyjmuje listę serwerów — agent sam wybiera
> który serwis wywołać na podstawie pytania. To nie jest hardcoded routing."*

### Oczekiwane wyniki
- Agent widzi 16 narzędzi (3 UC Functions + 13 Google Drive)
- Po zalogowaniu OAuth: agent tworzy Google Doc z profilem klienta
- Trace w MLflow pokazuje cały flow: UC Function → format → google_file_create

## Akt 4: Rejestracja i weryfikacja modelu (~15 min)

### Kontekst fabularny
> *CTO: "Agent działa w notebooku — ale jak go wdrożymy? Potrzebuję go jako
> zarejestrowany model w Unity Catalog, gotowy do deployment. I bez wpisywania numerów wersji na sztywno."*

### Co robimy
| Komórka | Co | Wynik |
| --- | --- | --- |
| 22 | Export: `retail_agent.py` (pyfunc, models from code) + config JSON → Volume | Kod agenta jako artefakt |
| 23 | `mlflow.pyfunc.log_model()` z `resources` + `register_model()` + **alias `@champion`** | `workspace.default.retail_customer_agent@champion` |
| 24 | Załadowanie **przez alias** + `predict()` | Weryfikacja end-to-end |

### Punkt dyskusji
> *"Model w Unity Catalog ma:
> - **Wersjonowanie** — każda zmiana = nowa wersja
> - **Alias @champion** — konsumenci (endpoint, WS2 monitoring) wskazują alias, nie numer; ten sam wzorzec co klasyfikator z WS1 i łańcuch RAG z WS3
> - **Governance** — kto ma dostęp (tak jak row filter / column mask z WS2!)
> - **Lineage** — z jakich danych, funkcji i endpointów korzysta (`resources`)
> - **Gotowość do serving** — Model Serving endpoint = 1 klik
>
> To zamyka pętlę czterech warsztatów:
> dane (WS1) → zabezpieczenia (WS2) → RAG (WS3) → agent (WS4)."*

### Oczekiwane wyniki
- Model `retail_customer_agent` zarejestrowany w Unity Catalog z aliasem `@champion`
- `predict()` zwraca poprawną odpowiedź
- Artefakty (kod + config) w Volume

---

### Podsumowanie Aktów 1–4

Zbudowaliśmy kompletną AI aplikację na danych retail:
`gold_customer_360 → UC Functions (bez PII) → LangChain Agent + guardrails → MLflow Tracing → UC Model @champion`

**Pytania do dyskusji:**
1. Dlaczego guardrail PII jest w FUNKCJI (nie zwraca tax_id), a nie tylko w promptcie? (Odp: prompt można obejść — §4b w WS2; narzędzia nie)
2. Czy agent potrzebuje guardrails z WS2? (Odp: tak — system prompt + safety filter + AI Gateway na endpoincie w Akcie 5)
3. Jak monitorować jakość odpowiedzi agenta w produkcji? (Odp: inference table → pipeline z WS2 Cz. 4 §7)
4. Co jest potrzebne do wdrożenia jako Model Serving endpoint? (Odp: model w UC + alias — Akt 5)

## Akt 5: Wdrożenie — Serving, batch, App, inference table (~30 min)

### Kontekst fabularny
> *CTO: "Agent jest zarejestrowany w UC. Teraz chcę go w przeglądarce —
> interfejs webowy, który każdy w firmie może użyć bez notebooka. I żeby analitycy mogli
> odpytać go hurtowo z SQL. I żeby KAŻDE pytanie było zapisane — do monitoringu z WS2."*

### Co robimy
| Komórka | Co | Wynik |
| --- | --- | --- |
| 26 | Model Serving endpoint z wersji **`@champion`**; **AI Gateway: inference table + usage tracking** | Endpoint `workshop-retail-agent`, tabela `retail_agent_inference_payload` |
| 27 | Czekanie + test endpointu (Databricks SDK) | Odpowiedź z REST API |
| 28 | Ten sam endpoint przez **`mlflow.deployments`** | Ten sam payload, inny klient |
| 30 | **Batch inference w SQL**: `ai_query('workshop-retail-agent', named_struct('prompt', …))` → tabela Delta | 5 odpowiedzi, w tym odmowa na PII |
| 31 | `app.py` (Gradio) + `app.yaml` + `requirements.txt` → workspace | Kod aplikacji |
| 32 | `apps.create_and_wait` + `deploy_and_wait` + CAN_QUERY dla SP aplikacji | URL aplikacji |
| 34 | **Inference table**: rozpakowanie request/response agenta | Most do monitora Time Series z WS2 §7 |

### Punkt dyskusji
> *"Databricks App to kontener z kodem Gradio/Streamlit. Łączy się z endpointem
> przez SDK — użytkownik pisze pytanie, app wysyła do Model Serving,
> agent wywołuje UC Functions i zwraca odpowiedź. Cały łańcuch jest
> monitorowany przez MLflow Tracing, a każdy request ląduje w inference table.
>
> Ten sam endpoint obsługuje App (online) i `ai_query` (batch) — jeden model, jedna polityka.
> Nowa wersja agenta? Rejestrujesz v2, przepinasz alias @champion, `update_config` — App nie wie o zmianie.
>
> To zamyka pełną pętlę czterech warsztatów:
> dane (WS1) → zabezpieczenia (WS2) → RAG (WS3) → aplikacja (WS4) → monitoring (z powrotem do WS2)."*

### Oczekiwane wyniki
- Model Serving endpoint READY (wersja z `@champion`), AI Gateway włączony
- Test z notebooka zwraca poprawną odpowiedź (SDK i mlflow.deployments)
- Tabela `retail_agent_batch_answers` z 5 odpowiedziami (pytanie o tax_id → odmowa)
- Databricks App dostępna pod URL
- Inference table (po kilku minutach) z payloadami agenta

# Podsumowanie: pełen obraz

## Co zbudowaliśmy w 4 warsztaty

```
┌───────────────────────────────────────────────────────────────┐
│  DATABRICKS MARKETPLACE                               │
│  customers (28K) + sales_orders (4K) + sales (360)    │
└───────────────────────────┬───────────────────────────────────┘
                            │
        ┌───────────────┴───────────────┐
        │  GOLD_CUSTOMER_360          │
        │  28,813 wierszy × 19 kolumn  │
        │  🔒 PII: tax_id, name         │
        └────┬───────┬───────┬───────┬───┘
             │       │       │       │
    ┌───────┴┐ ┌────┴─┐ ┌───┴──┐ ┌─┴──────┐
    │ ML     │ │ Genie│ │ Dash │ │ Progn. │
    │ Model  │ │ Space│ │ board│ │ 14 dni │
    └────┬───┘ └──┬───┘ └──────┘ └────────┘
         │        │
    ┌────┴───────┴───────────────────────┐
    │  ZABEZPIECZENIA (WS2 Cz. 1-2)        │
    │  🛡️ System prompt  🔒 Row filter      │
    │  🛡️ Safety filter  🔒 Column mask     │
    └────────────┬────────────────────┘
                 │
    ┌────────────┴────────────────────┐
    │  EWALUACJA (WS2 Cz. 3)              │
    │  ✅ Gold quality: 7/7 PASS           │
    │  ✅ Genie eval: safety=100%           │
    │  ✅ no_pii_leak: 100%                 │
    └────────────┬────────────────────┘
                 │
    ┌────────────┴────────────────────┐
    │  MONITORING (WS2 Cz. 4)             │
    │  📈 Profil danych                    │
    │  ⚠️ Dryf (VIP z-score=5.36)          │
    │  🤖 Model quality z WS1              │
    │  🔒 Audit guardrails                 │
    └─────────────────────────────────┘
```

## Funkcjonalności Databricks użyte w case study

| Funkcjonalność | Gdzie | Po co |
| --- | --- | --- |
| **Marketplace** | WS1 S1 | Pozyskanie gotowego datasetu + **przegląd licencji, DESCRIBE CATALOG (Delta Sharing)** |
| **Unity Catalog** | Wszędzie | 3-poziomowa hierarchia, governance |
| **SQL w notebooku** | WS1 S1-3 | Eksploracja, trendy, rankingi |
| **AI Functions** | WS1 S3 | ai_query, ai_classify w SQL |
| **PySpark** | WS1 S4 | JSON parsing, RFM, feature engineering |
| **Delta Lake** | WS1 S5 | ACID, Time Travel, wersjonowanie |
| **MLflow** | WS1 S6-8 | Autolog, tracking, model registry, **alias `@champion`** |
| **AutoML** | WS1 S8b | Automatyczny dobór modelu |
| **Dashboard SDK** | WS1 S10 | Wizualizacja dla biznesu |
| **Genie Space** | WS1 S10 | AI asystent NLP nad danymi |
| **System Prompt** | WS2 Cz.1 | Guardrail na poziomie LLM |
| **Safety Filter** | WS2 Cz.1 | Blokowanie szkodliwych zapytań |
| **Własny guard (wzór Llama Guard)** | WS2 Cz.1 §6 | Taksonomia unsafe S1–S6, werdykt pre/post-call |
| **AI Gateway + Secret scope** | WS2 Cz.1 §7 | Guardrails, PII block, inference table na endpoincie; PAT w sekretach |
| **Row Filter** | WS2 Cz.2 | Ograniczenie widoczności wierszy |
| **Column Mask** | WS2 Cz.2 | Maskowanie PII (tax_id) |
| **mlflow.genai.evaluate()** | WS2 Cz.3 | Ewaluacja Genie Space + Gold Table |
| **Benchmark A/B (ROUGE-1, sędzia 1–5)** | WS2 Cz.3 §3 | Baseline vs challenger, mean/variance, log inferencji |
| **Lakehouse Monitoring (Snapshot)** | WS2 Cz.4 | Profil, dryf, dashboard automatyczny |
| **Monitoring odpowiedzi LLM (Time Series)** | WS2 Cz.4 §7 | Payloady → toxicity/readability/refusal → monitor 5 min |
| **fpdf2 → PDF** | WS3 Cz. 1 | 10 artykułów PDF z danych strukturalnych |
| **ai_parse_document (2.0)** | WS3 Cz. 2 | Parsowanie PDF, metadane, obrazy stron, render bbox |
| **Chunking (LangChain)** | WS3 Cz. 2 | RecursiveCharacterTextSplitter 2000/200, tabela chunków |
| **Vector Search** | WS3 Cz. 2 | Delta Sync index na chunkach; ANN / hybrid / full-text / filtry / reranking |
| **LangChain RAG chain w UC** | WS3 Cz. 2 | Tracing + `mlflow.langchain.log_model` + `@champion` |
| **Knowledge Assistant** | WS3 Cz. 3 | Chatbot RAG na dokumentach z UC Volume (Agent Bricks) |
| **mlflow.genai.evaluate() (KA)** | WS3 Cz. 4 | Ewaluacja KA tymi samymi scorerami co WS2 |
| **Tool Calling (UC Function + OpenAI API)** | WS1 S3b | `get_revenue_summary` — LLM sam wywołuje UC Function; most do WS4 |
| **UC Functions** | WS4 Akt 2 | SQL + Python jako narzędzia agenta; test `execute_function`; AI Playground Tools |
| **MCP Google Drive** | WS4 Akt 3b | 13 narzędzi Google (Docs/Sheets/Slides) przez Model Context Protocol; agent LangGraph |
| **UCFunctionToolkit** | WS4 Akt 3 | LangChain agent z narzędziami UC |
| **MLflow Tracing** | WS4 Akt 3 | Autolog, tagi, walidacja promptu, trace'y w Unity Catalog |
| **Model Registry (pyfunc)** | WS4 Akt 4 | Rejestracja agenta w Unity Catalog |
| **Model Serving** | WS4 Akt 5 | Endpoint z `@champion`, AI Gateway inference table, `mlflow.deployments` |
| **ai_query na własnym endpoincie** | WS4 Akt 5 | Batch inference agenta w SQL → Delta |
| **Databricks Apps** | WS4 Akt 5 | Interfejs webowy Gradio dla agenta |

## Pytania do dyskusji na zakończenie

1. **Warstwowość:** Ile warstw ochrony ma nasz system? (Odp: 7 — prompt, safety filter, własny guard, AI Gateway, row filter, column mask, scorer/monitoring)
2. **Dryf:** Co zrobić gdy z-score VIP > 2? (Odp: alert → retrain → redeploy)
3. **Produkcja:** Czego brakuje do wdrożenia? (Odp: scheduler, CI/CD, ABAC policies, scoring na żywo)
4. **Regulacje:** Czy ten system spełnia RODO? (Odp: column mask + row filter = pseudonimizacja)
5. **Agent app:** Co trzeba dodać z WS2 do agenta z WS4? (Odp: AI Gateway na endpoincie `workshop-retail-agent` + monitor Time Series na jego inference table — pipeline z WS2 §7)
6. **Koszt vs jakość:** Kiedy zmienić Llama 70B na tańszy model? (Odp: gdy benchmark z WS2 §3 pokaże porównywalną średnią sędziego i niską wariancję)
7. **MCP:** Jak dodać kolejne źródła zewnętrzne? (Odp: `DatabricksMultiServerMCPClient` + serwery MCP: Genie, AI Search, Slack, Jira — agent sam wybiera który serwis wywołać)

# Linki do materiałów

| Materiał | Opis |
| --- | --- |
| **Warsztat 1** | [Retail Forecasting Workshop od danych do AI](#notebook-3153402175013847) |
| **Warsztat 2** | [Retail Workshop 2 Guardrails Monitoring Ewaluacja](#notebook-3153402175013848) |
| **Warsztat 3** | [Retail Workshop 3 RAG i Knowledge Assistant](#notebook-3781019916085186) |
| **Warsztat 4** | [Retail Workshop 4 Agent App](#notebook-3781019916085187) |
| **Ten przewodnik** | [Retail Workshop Scenariusz i Przewodnik](#notebook-3781019916085188) |

### Wymagania dla uczestników
- Databricks workspace z Unity Catalog
- Dostęp do Marketplace (dataset: https://adb-7405615837166522.2.azuredatabricks.net/marketplace/consumer/listings/a82597f6-5ada-49d5-b934-d6c9dece16a1?o=7405615837166522 - `databricks_simulated_retail_customer_data`)
- Serverless compute
- Endpointy `databricks-meta-llama-3-3-70b-instruct`, `databricks-gpt-oss-20b` (challenger w WS2), `databricks-gte-large-en` (embeddingi w WS3)
- Schema `workspace.default` z uprawnieniami CREATE TABLE, CREATE FUNCTION, CREATE MODEL, CREATE VOLUME
- Do WS2 §7 i WS4 Akt 5: prawo tworzenia Model Serving endpointów i secret scope (opcjonalne — sekcje mają ścieżkę zapasową)

### Przygotowanie prowadzącego
1. Uruchom WS1 w całości przed warsztatem (\~15 min) — tworzy Gold Table i model z aliasem `@champion` (WS2 i WS4 tego wymagają)
2. Uruchom WS2 Część 1–2 (guardrails) + Część 3 (ewaluacja i benchmark) — \~30 min; jeśli robisz §7 (AI Gateway), wygeneruj PAT wcześniej
3. Monitoring (WS2 Część 4) wymaga \~10 min na refresh monitora Snapshot i kolejne kilka minut na monitor Time Series (§7c) — rozważ `WAIT_FOR_REFRESH = False` na żywo
4. Przygotuj screenshoty MLflow UI i Catalog Explorer jako backup
5. WS3 RAG: uruchom Część 1–2 wcześniej (parsowanie + chunking + indeks to \~10 min); sprawdź, że Vector Search endpoint `retail-rag-vs` jest ONLINE i pakiety `databricks-langchain`/`langchain-text-splitters` instalują się bez konfliktów
6. WS4: uruchom instalację `unitycatalog-ai[databricks]` przed startem lub zarezerwuj ~2 min na restart Pythona
7. WS4: endpoint `workshop-retail-agent` startuje \~10 min — utwórz go przed warsztatem albo zaplanuj przerwę; payloady inference table pojawiają się z opóźnieniem, więc komórkę z inference table pokazuj na końcu
8. Pakiety Python instalowane w WS2 (`rouge-score`, `textstat`) i WS3 — przetestuj instalację na docelowym środowisku Serverless dzień wcześniej